# While Foundry is working

Let's use the wait to practice the decisions that make an AI system trustworthy. Pick one challenge, then return to the core notebook when its job finishes. These are optional; none is required to continue the climb.

**Run this notebook in its own kernel.** It uses Python's standard library, reads local files, makes no Azure calls and writes nothing. Do not restart the busy core kernel or change its datasets, rubric or thresholds.

| Challenge | A useful moment | What we practice |
|---|---|---|
| Manual detective | Practice calls or trace ingestion | Finding a problem in the answer key, not just the agent |
| Judge the judge | Rubric generation or baseline eval | Asking whether a judge has enough evidence |
| The budget hearing | Optimizer job | Explaining why unknown cost is not a saving |
| Route detective | Router evaluation | Spotting missing answers before comparing averages |

In [ ]:
import json
from pathlib import Path
from collections import Counter

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data" / "practice.jsonl").is_file()), None)
if root is None:
    raise FileNotFoundError("Open this notebook from the LAB663 repository.")
practice = [json.loads(line) for line in (root / "data" / "practice.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
print("Ready: local files only. Our core job can keep running.")

## 1. Manual detective

The agent says a backpack's manual contradicts itself. Our answer key says there is no rain cover. Who needs fixing? Let's look at the evidence before blaming the model.

**Business lens:** a misleading product promise can cause returns and lost trust. **Engineering lens:** a quote validator proves that text exists, not that our expected answer handles every conflicting statement. Predict what we will find.

In [ ]:
case = next(case for case in practice if case["id"] == "pr-04")
print("Question:", case["query"])
print("Answer key:", case["expected_behavior"])
manual = (root / "data" / "manuals" / "product_info_2.md").read_text(encoding="utf-8")
for number, line in enumerate(manual.splitlines(), 1):
    if "rain cover" in line.lower():
        print(f"Line {number}: {line}")

**Expected:** the manual includes both a rain-cover claim and a specification saying no. Write one customer-facing sentence that acknowledges the conflict without inventing an answer.

**What happened:** this is a data-quality issue to record for the next dataset version. Do not edit `practice.jsonl` during the current comparison. If we change the answer key, we must remeasure the baselines. For an intermediate challenge, explain why validating verbatim quotes did not catch this contradiction.

## 2. Judge the judge

A judge is asked whether an answer is grounded in a manual, but receives only the answer. Can it know? These two deliberately written examples let us separate a confident tone from a supported claim. They are not live model outputs.

Read the manual excerpt, then choose which answer deserves the better score. What evidence would a retrieval-quality criterion need in addition to the final answer?

In [ ]:
case = next(case for case in practice if case["id"] == "pr-03")
print("Question:", case["query"])
for fact in case["facts"]:
    print("Evidence:", fact["quote"])
examples = {
    "A": "The jacket is water-resistant, not waterproof.",
    "B": "Yes, it is fully waterproof, so you can rely on it in any storm.",
}
for label, answer in examples.items():
    print(f"{label}: {answer}")

**Expected:** A preserves the qualification; B overclaims. Neither final answer alone shows how a search query was formed. Name the extra evidence needed: the tool call, its arguments and the retrieved results.

**Why it matters:** a weak judge can reward polished mistakes or punish correct answers whose supporting evidence was hidden. In the core notebook, inspect the actual per-dimension reasons before interpreting an aggregate. Do not change the frozen rubric in this exercise.

## 3. The budget hearing

Let's play the decision maker. A team says, "The router used fewer tokens, so approve the saving." Our job is to ask what would turn that statement into a defensible estimate.

Read the price file, not an imagined price list. Can we price every serving model in the right region, including cached input? The optimizer's job estimate is a different budget from the deployed agent's ongoing bill.

In [ ]:
pricing = json.loads((root / "data" / "pricing.json").read_text(encoding="utf-8"))
print("Region:", pricing["region"], "| Price date:", pricing.get("as_of"), "| Source:", pricing.get("source_url"))
for model, rates in pricing["models"].items():
    missing = [name for name in ("input_per_million", "output_per_million", "cached_input_per_million") if rates.get(name) is None]
    print(model, "missing: " + ", ".join(missing) if missing else "rates supplied; verify source and region")
print("Decision template: Hold or proceed? Evidence I still need: ...")

**Expected:** unfilled prices are clearly visible. No dollar amount is invented. A defensible decision also needs complete request coverage, acceptable quality, per-model usage with the judge separated, and relevant traffic volume.

**Go deeper:** cached input, multi-step tool use and changing serving models can all move cost. Explain which charges our model-only estimate excludes, and why a global deployment's resource region is not a promise about processing location.

## 4. Route detective

A batch can finish while some requests fail. Let's inspect a completed local evidence file and distinguish **failed the rubric** from **never received a score**. This does not contact Foundry or start a new run.

If the first core batch is still running, there may be no evidence file yet. Return later, or discuss why removing a failed question would make the comparison unfair.

In [ ]:
paths = sorted((root / ".lab-results").glob("*.json"), key=lambda path: path.stat().st_mtime, reverse=True)
evidence = None
for path in paths:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if "runs" in payload:
        evidence = payload
        print("Reading:", path.name, "|", payload["label"])
        break
if evidence is None:
    print("No completed evidence file yet. Let the core job continue; do not resubmit it.")
else:
    for repeat in evidence["runs"]:
        run = repeat["run"]
        counts = run["result_counts"]
        print(run["id"], run["status"], counts)
        models = Counter(item["sample"]["model"] for item in repeat["items"]
                         if item.get("sample") and not item["sample"].get("error"))
        print("Final serving models:", dict(models))
        for item in repeat["items"]:
            sample = item.get("sample") or {}
            if sample.get("error"):
                print("Unscored case:", item["datasource_item"].get("id"), sample["error"])


**Expected:** either an honest "not ready yet" message or real run counts and serving-model counts. A rubric failure is still a measured answer. An errored or skipped item is not; the core helper marks that batch invalid rather than averaging the remaining questions.

**What did we learn?** Before changing a routing mode, distinguish reliability, answer quality, judge inputs and usage attribution. Keep the same question set. Then return to the waiting core cell and read its status; a finished bonus challenge does not mean the service job finished.